# LoRA fine-tuning of BERT on a GLUE task: Kaggle feasibility test

**Goal:** find out whether we can fine-tune a BERT model with LoRA on Kaggle's free GPU, how long it takes, and how much GPU memory it needs.

**Big picture (what happens in this notebook):**
1. **Setup**: install `peft`, import libraries, check the GPU
2. **Config**: all settings in ONE cell (task, model, LoRA, training)
3. **Data**: load a GLUE task from Hugging Face
4. **Tokenize**: turn text into numbers BERT understands
5. **Model + LoRA**: load BERT, freeze it, attach small trainable LoRA adapters
6. **Train**: fine-tune with the Hugging Face `Trainer`, measuring time and memory
7. **Evaluate + feasibility report**: accuracy, time, memory, and an estimate for a full run
8. **Save**: store the small LoRA adapter

**Before you run:** in the Kaggle right sidebar set **Accelerator = GPU (T4 or P100)** and **Internet = On** (needed to download the model, the dataset and `peft`).

## 1. Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # use ONE gpu, so time/memory numbers are clean
os.environ["WANDB_DISABLED"] = "true"         # no Weights&Biases login prompt
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# PROBLEM WE FACED:
#   Kaggle comes with an old "torchao" (0.10.0) preinstalled. Installing peft can pull in a newer
#   "transformers", which refuses to import with: "ImportError: Found an incompatible version of
#   torchao. Found version 0.10.0, but only versions above 0.16.0 are supported".
# WHY THIS FIX:
#   Our notebook does not use torchao (it is only needed for model quantization), so we simply
#   remove it. That makes transformers skip the version check.
# NOTE:
#   If the error still shows up, the old version is still loaded in memory. Restart the session
#   (Run -> Restart session) and run this cell first, before any other import.
!pip uninstall -y torchao
!pip install -q peft

# FALLBACK (only if removing torchao causes a different error): upgrade it instead of removing it
# !pip install -q -U "torchao>=0.16.0"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import time, json, math, inspect
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef
import transformers, peft

print("torch:", torch.__version__, "| transformers:", transformers.__version__, "| peft:", peft.__version__)
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print("GPU:", torch.cuda.get_device_name(0),
          "| memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("WARNING: no GPU found. Turn on a GPU accelerator in the Kaggle sidebar.")

torch: 2.11.0+cu128 | transformers: 5.16.1 | peft: 0.20.0
GPU: Tesla T4 | memory: 15.6 GB


## 2. Config: the only cell you need to edit

- **`TASK`**: which GLUE task. Sizes of the training sets (approx.): `rte` 2.5k, `mrpc` 3.7k, `cola` 8.5k, `sst2` 67k, `qnli` 105k, `qqp` 364k, `mnli` 393k.
- **`MAX_TRAIN_SAMPLES`**: use a small number (e.g. 2000) for a quick feasibility test; set `None` for the full training set.
- **`USE_LORA`**: `True` = LoRA, `False` = normal full fine-tuning (handy to compare time and memory).
- **LoRA settings**: `LORA_R` is the adapter "rank" (bigger = more trainable parameters), `LORA_ALPHA` is a scaling factor (commonly 2x the rank).

In [3]:
# ---------- Task and model ----------
TASK = "sst2"                       # cola, sst2, mrpc, qqp, qnli, rte, mnli
MODEL_NAME = "bert-base-uncased"    # e.g. "distilbert-base-uncased" for a smaller model

# ---------- Data size (for the quick test) ----------
MAX_TRAIN_SAMPLES = None            # None = full training set
MAX_EVAL_SAMPLES = None             # None = full validation set
MAX_LENGTH = 128                    # longer texts are cut to this many tokens

# ---------- LoRA ----------
USE_LORA = True
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["query", "value"]   # the attention layers of BERT that get adapters

# ---------- Training ----------
EPOCHS = 3                          # use 3 for a real run
BATCH_SIZE = 32
LEARNING_RATE = 2e-4 if USE_LORA else 2e-5   # LoRA needs a higher learning rate than full fine-tuning
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06
USE_FP16 = HAS_GPU                  # mixed precision: faster and uses less memory on GPU
SEED = 42
OUTPUT_DIR = "/kaggle/working/outputs"

## 3. Load the GLUE data

GLUE tasks differ in their columns: some have one sentence, some have two (a pair). The table below tells the tokenizer which columns to use.

In [4]:
# task -> (first text column, second text column or None)
TASK_COLUMNS = {
    "cola": ("sentence", None),
    "sst2": ("sentence", None),
    "mrpc": ("sentence1", "sentence2"),
    "qqp":  ("question1", "question2"),
    "qnli": ("question", "sentence"),
    "rte":  ("sentence1", "sentence2"),
    "mnli": ("premise", "hypothesis"),
}
assert TASK in TASK_COLUMNS, f"TASK must be one of {list(TASK_COLUMNS)}"
TEXT_A, TEXT_B = TASK_COLUMNS[TASK]
NUM_LABELS = 3 if TASK == "mnli" else 2
EVAL_SPLIT = "validation_matched" if TASK == "mnli" else "validation"

raw = load_dataset("nyu-mll/glue", TASK)
print(raw)

train_raw = raw["train"]
eval_raw = raw[EVAL_SPLIT]
FULL_TRAIN_SIZE = len(train_raw)          # remember it for the time estimate later

if MAX_TRAIN_SAMPLES is not None:
    train_raw = train_raw.shuffle(seed=SEED).select(range(min(MAX_TRAIN_SAMPLES, len(train_raw))))
if MAX_EVAL_SAMPLES is not None:
    eval_raw = eval_raw.select(range(min(MAX_EVAL_SAMPLES, len(eval_raw))))

print("\nExample:", train_raw[0])
print(f"Using {len(train_raw)} training and {len(eval_raw)} evaluation examples (full train set: {FULL_TRAIN_SIZE})")

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})

Example: {'sentence': 'hide new secretions from the parental units ', 'label': 0, 'idx': 0}
Using 67349 training and 872 evaluation examples (full train set: 67349)


## 4. Tokenize

BERT cannot read text, only numbers. The tokenizer splits text into tokens and maps each to an ID. We do **not** pad everything to the same length here. The data collator pads each batch only as much as needed, which saves time.

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    if TEXT_B is None:
        return tokenizer(batch[TEXT_A], truncation=True, max_length=MAX_LENGTH)
    return tokenizer(batch[TEXT_A], batch[TEXT_B], truncation=True, max_length=MAX_LENGTH)

def prepare(ds):
    ds = ds.map(tokenize, batched=True)
    drop = [c for c in ds.column_names if c not in ("input_ids", "attention_mask", "token_type_ids", "label")]
    return ds.remove_columns(drop)

train_ds = prepare(train_raw)
eval_ds = prepare(eval_raw)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print("Decoded example:", tokenizer.decode(train_ds[0]["input_ids"]))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 67349
})
Decoded example: [CLS] hide new secretions from the parental units [SEP]


## 5. Model + LoRA

**What LoRA does:** instead of updating all ~110 million BERT weights, we **freeze** BERT and add tiny trainable matrices ("adapters") next to the attention layers. Only those adapters (plus the new classification layer on top) are trained. That is why LoRA needs less memory and produces a very small file.

In [6]:
set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

if USE_LORA:
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,        # sequence classification
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
    )
    model = get_peft_model(model, lora_config)

def count_params(m):
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

TOTAL_PARAMS, TRAINABLE_PARAMS = count_params(model)
print(f"Total parameters:     {TOTAL_PARAMS:,}")
print(f"Trainable parameters: {TRAINABLE_PARAMS:,} ({100 * TRAINABLE_PARAMS / TOTAL_PARAMS:.2f}%)")

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Total parameters:     109,780,228
Trainable parameters: 296,450 (0.27%)


## 6. Training setup

We use the Hugging Face `Trainer`, which runs the training loop for us (batching, GPU, mixed precision, evaluation). `compute_metrics` turns the model's outputs into the usual GLUE scores.

In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    out = {"accuracy": accuracy_score(labels, preds)}
    if TASK in ("mrpc", "qqp"):
        out["f1"] = f1_score(labels, preds)
    if TASK == "cola":
        out["matthews"] = matthews_corrcoef(labels, preds)
    return out

steps_per_epoch = math.ceil(len(train_ds) / BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS

# the argument was renamed in newer transformers versions; pick whichever exists
_ta = inspect.signature(TrainingArguments.__init__).parameters
eval_kw = {"eval_strategy" if "eval_strategy" in _ta else "evaluation_strategy": "epoch"}

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=int(WARMUP_FRACTION * total_steps),
    fp16=USE_FP16,
    logging_steps=50,
    save_strategy="no",              # don't save checkpoints (saves disk and time); we save the adapter at the end
    report_to="none",
    seed=SEED,
    label_names=["labels"],
    remove_unused_columns=False,     # we already removed the unneeded columns ourselves
    dataloader_num_workers=2,
    **eval_kw,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print(f"{steps_per_epoch} steps per epoch, {total_steps} steps in total")

2105 steps per epoch, 6315 steps in total


## Testing Before Training

In [8]:
# Accuracy BEFORE training.
# Why: the classifier head is randomly initialized, so we expect roughly chance-level
# accuracy (~50% for 2 classes, ~33% for 3). If it is already high, something is wrong.
# If it is low before and high after training, we know the training really did something.

# We evaluate on a fixed subset of the training data (full train sets can be huge and slow to evaluate)
N_TRAIN_CHECK = min(2000, len(train_ds))
train_check_ds = train_ds.select(range(N_TRAIN_CHECK))

def evaluate_both(tag):
    train_m = trainer.evaluate(eval_dataset=train_check_ds, metric_key_prefix="train")
    val_m = trainer.evaluate(eval_dataset=eval_ds, metric_key_prefix="val")
    return {
        f"{tag}_train_acc": train_m["train_accuracy"], f"{tag}_train_loss": train_m["train_loss"],
        f"{tag}_val_acc": val_m["val_accuracy"], f"{tag}_val_loss": val_m["val_loss"],
    }

before = evaluate_both("before")
print("BEFORE training:", {k: round(v, 4) for k, v in before.items()})

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.687433,0,0.557000


Training Loss,Validation Loss,Epoch,Accuracy
No log,0.691378,0,0.510321


BEFORE training: {'before_train_acc': 0.557, 'before_train_loss': 0.6874, 'before_val_acc': 0.5103, 'before_val_loss': 0.6914}


## 7. Train (and measure time + GPU memory)

In [9]:
if HAS_GPU:
    torch.cuda.reset_peak_memory_stats()

start = time.time()
train_result = trainer.train()
TRAIN_SECONDS = time.time() - start

PEAK_GPU_GB = torch.cuda.max_memory_allocated() / 1e9 if HAS_GPU else 0.0
print(f"\nTraining finished in {TRAIN_SECONDS:.0f} s ({TRAIN_SECONDS / 60:.1f} min)")
print(f"Peak GPU memory: {PEAK_GPU_GB:.2f} GB")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.244543,0.242193,0.905963
2,0.188130,0.238261,0.907110
3,0.188500,0.241080,0.903670



Training finished in 359 s (6.0 min)
Peak GPU memory: 1.32 GB


## 8. Evaluate + feasibility report

In [10]:
metrics = trainer.evaluate()
print("Validation metrics:", {k: round(v, 4) for k, v in metrics.items() if isinstance(v, float)})

samples_processed = len(train_ds) * EPOCHS
sec_per_sample = TRAIN_SECONDS / samples_processed
full_run_hours = sec_per_sample * FULL_TRAIN_SIZE * 3 / 3600     # estimate: full train set, 3 epochs

print("\n" + "=" * 55)
print("FEASIBILITY REPORT")
print("=" * 55)
print(f"Task / model:              {TASK} / {MODEL_NAME}")
print(f"Method:                    {'LoRA (r=%d)' % LORA_R if USE_LORA else 'full fine-tuning'}")
print(f"Trainable parameters:      {TRAINABLE_PARAMS:,} of {TOTAL_PARAMS:,} ({100 * TRAINABLE_PARAMS / TOTAL_PARAMS:.2f}%)")
print(f"Training examples used:    {len(train_ds)} x {EPOCHS} epoch(s)")
print(f"Training time:             {TRAIN_SECONDS / 60:.1f} min")
print(f"Throughput:                {samples_processed / TRAIN_SECONDS:.1f} examples/sec")
print(f"Peak GPU memory:           {PEAK_GPU_GB:.2f} GB")
print(f"ESTIMATE full run:         ~{full_run_hours:.2f} h for {FULL_TRAIN_SIZE} examples x 3 epochs")
print("=" * 55)
print("(The estimate extrapolates from this run; real time varies a bit. Kaggle sessions last ~12 h max and the weekly GPU quota is limited, so check your quota in Kaggle.)")

report = {
    "task": TASK, "model": MODEL_NAME, "use_lora": USE_LORA, "lora_r": LORA_R,
    "total_params": TOTAL_PARAMS, "trainable_params": TRAINABLE_PARAMS,
    "train_examples_used": len(train_ds), "epochs": EPOCHS, "batch_size": BATCH_SIZE,
    "train_seconds": TRAIN_SECONDS, "examples_per_sec": samples_processed / TRAIN_SECONDS,
    "peak_gpu_gb": PEAK_GPU_GB, "estimated_full_run_hours_3_epochs": full_run_hours,
    "metrics": {k: v for k, v in metrics.items() if isinstance(v, float)},
}
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, "feasibility_report.json"), "w") as f:
    json.dump(report, f, indent=2)

Training Loss,Validation Loss,Epoch,Accuracy
0.188500,0.241080,3,0.903670


Validation metrics: {'eval_loss': 0.2411, 'eval_accuracy': 0.9037}

FEASIBILITY REPORT
Task / model:              sst2 / bert-base-uncased
Method:                    LoRA (r=8)
Trainable parameters:      296,450 of 109,780,228 (0.27%)
Training examples used:    67349 x 3 epoch(s)
Training time:             6.0 min
Throughput:                562.1 examples/sec
Peak GPU memory:           1.32 GB
ESTIMATE full run:         ~0.10 h for 67349 examples x 3 epochs
(The estimate extrapolates from this run; real time varies a bit. Kaggle sessions last ~12 h max and the weekly GPU quota is limited, so check your quota in Kaggle.)


## 9. Save the adapter

With LoRA, only the small adapter (a few MB) is saved, not the whole BERT model. To use it later, load the base model again and attach the adapter with `PeftModel.from_pretrained`.

In [11]:
SAVE_DIR = os.path.join(OUTPUT_DIR, "lora_adapter" if USE_LORA else "full_model")
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
size_mb = sum(os.path.getsize(os.path.join(SAVE_DIR, f)) for f in os.listdir(SAVE_DIR)) / 1e6
print(f"Saved to {SAVE_DIR} ({size_mb:.1f} MB)")

Saved to /kaggle/working/outputs/lora_adapter (1.9 MB)


In [12]:
print(model)

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): BertForSequenceClassification(
      (bert): BertModel(
        (embeddings): BertEmbeddings(
          (word_embeddings): Embedding(30522, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): L

## Testing After Training

In [13]:
# Accuracy AFTER training, compared with BEFORE
after = evaluate_both("after")

print(f"\n{'':<22}{'before':>10}{'after':>10}")
print("-" * 42)
for name, key in [("Train accuracy", "train_acc"), ("Train loss", "train_loss"),
                  ("Validation accuracy", "val_acc"), ("Validation loss", "val_loss")]:
    print(f"{name:<22}{before['before_' + key]:>10.4f}{after['after_' + key]:>10.4f}")

# Quick interpretation
if after["after_val_acc"] > before["before_val_acc"] + 0.1:
    print("\nOK: validation accuracy clearly improved, so the training worked.")
else:
    print("\nWARNING: little or no improvement. Check the learning rate, labels, or number of epochs.")

gap = after["after_train_acc"] - after["after_val_acc"]
if gap > 0.1:
    print(f"Note: train accuracy is {gap:.1%} higher than validation, which suggests overfitting.")

Training Loss,Validation Loss,Epoch,Accuracy
0.188500,0.146073,3,0.946500


Training Loss,Validation Loss,Epoch,Accuracy
0.188500,0.241080,3,0.903670



                          before     after
------------------------------------------
Train accuracy            0.5570    0.9465
Train loss                0.6874    0.1461
Validation accuracy       0.5103    0.9037
Validation loss           0.6914    0.2411

OK: validation accuracy clearly improved, so the training worked.


## What to do next

1. Run once as-is (2000 examples, 1 epoch) to see if everything works and how fast it is.
2. Set `MAX_TRAIN_SAMPLES = None` and `EPOCHS = 3` for a real run (or compare against `USE_LORA = False`).
3. Copy the numbers from the feasibility report into your notes. They are useful for the paper.

---
# NEW part: grid search for the default LoRA values

Everything from here on is new, the cells above are unchanged. The new part reuses the config (section 2), the tokenizer and data collator (section 4) and the `Trainer` setup (section 6).

**What is trained in every run:** only the small LoRA matrices A, B and the new classifier layer. BERT itself stays frozen.

**To skip the SST-2 run above:** run the cells up to "6. Training setup", then continue here.

## 10. NEW Study config: the only new cell you need to edit

- **`STUDY_QUICK`**: `True` = short test of all new cells (30 training steps, a few minutes). `False` = the real run.
- **`STUDY_DATASETS`**: which datasets. Remove one to save GPU time.
- **Sizes**: how many texts for training and for the dev split. The test split stays untouched for the paper.
- **Reference**: taken from the config cell above (`LORA_R`, `LORA_TARGET_MODULES`, `LEARNING_RATE`, `MAX_LENGTH`).

In [ ]:
# NEW 10: study config
import glob, re, shutil
import pandas as pd
import matplotlib.pyplot as plt
from datasets import Dataset
from sklearn.model_selection import train_test_split

import datasets
transformers.logging.set_verbosity_error()   # hide the long model loading reports
datasets.disable_progress_bars()             # hide the tokenizing progress bars, so the run output stays readable
transformers.utils.logging.disable_progress_bar()   # hide the model loading bars
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)   # library notices about future versions

if "trainer" in globals():  # free the GPU from the feasibility run above
    del trainer, model
    torch.cuda.empty_cache()

# ---------- 1. Size of the study ----------
STUDY_QUICK = True                  # True = short test of all new cells (30 steps), False = the real run
STUDY_DATASETS = ["sst2", "ag_news", "20newsgroups"]
STUDY_SEEDS = [0, 1] if STUDY_QUICK else [0, 1, 2]
STUDY_TRAIN_SAMPLES = 500 if STUDY_QUICK else 20000
STUDY_DEV_SAMPLES = 200 if STUDY_QUICK else 5000      # sst2 only has 436 dev sentences
STUDY_PROBE_SAMPLES = 200 if STUDY_QUICK else 2000    # kept apart for the XAI part
STUDY_DIR = os.path.join(OUTPUT_DIR, "study")

# ---------- 2. Training settings of the trainable parts (LoRA A, B + classifier layer) ----------
# Same for every run. BERT itself is frozen and gets no updates.
# Taken from the config cell above: BATCH_SIZE, WEIGHT_DECAY, WARMUP_FRACTION, USE_FP16
STUDY_EPOCHS = {"sst2": 2, "ag_news": 2, "20newsgroups": 4}   # 20newsgroups has fewer texts, so more epochs
STUDY_QUICK_STEPS = 30                                         # used instead of epochs when STUDY_QUICK

# ---------- 3. Reference run = the LoRA config cell above ----------
REFERENCE = dict(r=LORA_R, modules=LORA_TARGET_MODULES, layers=None,  # layers=None: all 12 blocks
                 lr=LEARNING_RATE, max_len=MAX_LENGTH, cut="right")    # cut="right" keeps the start of a text

# ---------- 4. The grid settings are added by the grid cell below ----------
def change(**kw):
    return {**REFERENCE, **kw}

STUDY_SETTINGS = {"reference": ("reference", REFERENCE)}
PUBLISHED = {"sst2": 0.929, "ag_news": 0.9475, "20newsgroups": 0.861}   # full fine-tuning: Devlin 2019, Sun 2019, Dai 2022

def what_changed(setting):
    """Short text: what differs from the reference, e.g. 'layers=[9, 10, 11], r=32'."""
    s = STUDY_SETTINGS[setting][1]
    diff = [f"{k}={v}" for k, v in s.items() if v != REFERENCE[k]]
    return ", ".join(diff) if diff else "nothing (reference)"

def runs_for(dataset):
    return [n for n, (g, _) in STUDY_SETTINGS.items() if g != "length" or dataset == "20newsgroups"]

def members(group):
    return [n for n, (g, _) in STUDY_SETTINGS.items() if g == group]

# continue an earlier run: on Kaggle add the output of an earlier version as input
earlier = glob.glob("/kaggle/input/**/study/results.csv", recursive=True)
if earlier and not os.path.exists(os.path.join(STUDY_DIR, "results.csv")):
    shutil.copytree(os.path.dirname(earlier[0]), STUDY_DIR, dirs_exist_ok=True)
os.makedirs(STUDY_DIR, exist_ok=True)

print(f"reference: r={LORA_R}, alpha={LORA_ALPHA}, modules={LORA_TARGET_MODULES}, all 12 layers, max length {MAX_LENGTH}")

## 11. NEW Load the three datasets

| dataset | task | classes | texts |
|---|---|---|---|
| SST-2 | sentiment | negative, positive | movie review sentences |
| AG News | topic | World, Sports, Business, Sci/Tech | news title + description |
| 20 Newsgroups | topic | 20 forum groups, e.g. comp.graphics, rec.autos, sci.space | forum posts, real documents |

**Dev split:** used for all our choices. SST-2 hides its test labels, so its validation sentences are split into dev and test. For the other two, dev comes from train.

The cell also shows one row per class (name, number of texts, one example) and a bar plot of texts per class.

In [ ]:
# NEW 11: load the three datasets
# Task in all three: one whole text -> one class. Accuracy and macro F1 compare the predicted class with the label.
#   sst2          sentiment of a movie review sentence: negative, positive
#   ag_news       topic of a news text (title + description): World, Sports, Business, Sci/Tech
#   20newsgroups  topic of a forum post (real documents): 20 groups, e.g. comp.graphics, rec.autos, sci.space
def load_study_data(name):
    if name == "sst2":
        ds = load_dataset("nyu-mll/glue", "sst2")
        train, val = (ds[s].to_pandas().rename(columns={"sentence": "text"}) for s in ("train", "validation"))
        dev, test = train_test_split(val, test_size=0.5, stratify=val.label, random_state=0)
        names = ["negative", "positive"]
    else:
        ds = load_dataset("fancyzhx/ag_news" if name == "ag_news" else "SetFit/20_newsgroups")
        train, test = ds["train"].to_pandas(), ds["test"].to_pandas()
        if name == "ag_news":
            names = ds["train"].features["label"].names
        else:
            names = [n for _, n in sorted(set(zip(train.label, train.label_text)))]
        test = test[test.text.str.strip() != ""]
        train = train[(train.text.str.strip() != "") & ~train.text.isin(test.text)].drop_duplicates("text")
        train, dev = train_test_split(train, test_size=min(STUDY_DEV_SAMPLES, len(train) // 6), stratify=train.label, random_state=0)
    train = train[~train.text.isin(dev.text)][["text", "label"]].sample(frac=1, random_state=0).reset_index(drop=True)
    dev = dev[["text", "label"]].sample(frac=1, random_state=0).head(STUDY_DEV_SAMPLES).reset_index(drop=True)
    return {"train": train[:-STUDY_PROBE_SAMPLES][:STUDY_TRAIN_SAMPLES].reset_index(drop=True),
            "probe": train[-STUDY_PROBE_SAMPLES:].reset_index(drop=True), "dev": dev,
            "test_size": len(test), "num_labels": train.label.nunique(), "class_names": names}

STUDY_DATA = {name: load_study_data(name) for name in STUDY_DATASETS}

def tokenized(dataset, split, s):
    """Hugging Face dataset with token ids, cut at max_len from the right (keeps the start) or the left (keeps the end)."""
    tokenizer.truncation_side = s["cut"]
    ds = Dataset.from_pandas(STUDY_DATA[dataset][split], preserve_index=False)
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=s["max_len"]), batched=True, remove_columns=["text"])

rows = []
for name, d in STUDY_DATA.items():
    tokens = np.array([len(t) for t in tokenizer(d["dev"].text.tolist()).input_ids])
    rows.append({"dataset": name, "classes": d["num_labels"],
                 "class_names": ", ".join(d["class_names"][:4]) + (", ..." if len(d["class_names"]) > 4 else ""), "train": len(d["train"]), "dev": len(d["dev"]),
                 "test_untouched": d["test_size"], "median_tokens": int(np.median(tokens)),
                 "above_128": f"{np.mean(tokens > 128):.0%}", "above_512": f"{np.mean(tokens > 512):.0%}"})
data_table = pd.DataFrame(rows)
data_table


# ---------- classes: one row per class, with counts and one example ----------
rows = []
for name, d in STUDY_DATA.items():
    for i, cls in enumerate(d["class_names"]):
        examples = d["dev"][d["dev"].label == i].text
        rows.append({"dataset": name, "label": i, "class": cls,
                     "train_texts": int((d["train"].label == i).sum()), "dev_texts": int((d["dev"].label == i).sum()),
                     "example": examples.iloc[0][:120].replace("\n", " ") if len(examples) else ""})
class_table = pd.DataFrame(rows)

# NEW plotting: texts per class in the full dataset (all splits, before we take our subsets)
FULL = {"sst2": ("nyu-mll/glue", "sst2", ["train", "validation"]), "ag_news": ("fancyzhx/ag_news", None, ["train", "test"]),
        "20newsgroups": ("SetFit/20_newsgroups", None, ["train", "test"])}
fig, axes = plt.subplots(1, len(STUDY_DATASETS), figsize=(5 * len(STUDY_DATASETS), 0.3 * max(len(STUDY_DATA[d]["class_names"]) for d in STUDY_DATASETS) + 2), squeeze=False)
for ax, name in zip(axes[0], STUDY_DATASETS):
    path, config, splits = FULL[name]
    ds = load_dataset(path, config)
    names = STUDY_DATA[name]["class_names"]
    y = np.arange(len(names))
    for k, split in enumerate(splits):
        counts = np.bincount(ds[split]["label"], minlength=len(names))
        ax.barh(y + (0.2 if k == 0 else -0.2), counts, height=0.4, label=f"{split} ({counts.sum():,})")
    ax.set(yticks=y, title=name, xlabel="number of texts")
    ax.set_yticklabels(names, fontsize=7)
    ax.legend(fontsize=7)
fig.tight_layout()
plt.rcParams["svg.fonttype"] = "none"
for ext in ("png", "svg"):
    fig.savefig(os.path.join(STUDY_DIR, f"texts_per_class.{ext}"), dpi=200, bbox_inches="tight")
plt.show()

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)
class_table

## 12. NEW One training run

**What happens in one run** (one setting, one dataset, one seed):
1. tokenize train and dev
2. build the model: BERT (frozen) + new classifier layer + LoRA where the setting says
3. train with the same `TrainingArguments` as section 6, only the setting's values change
4. **accuracy test** on the dev texts: predicted class vs. true class
5. save the small adapter and the prediction of every dev text

**How the scores are computed**, for $N$ dev texts with true class $y_i$ and predicted class $\hat y_i$:
- accuracy $= \frac{1}{N}\sum_i [\hat y_i = y_i]$, the share of right answers
- macro F1 $= \frac{1}{C}\sum_c F1_c$ with $F1_c = \frac{2 P_c R_c}{P_c + R_c}$, so every class counts the same, $R_c$ = right ones among the texts of class $c$, $P_c$ = right ones among the texts predicted as $c$

The last lines print the trainable weights per setting. The reference must give 296,450 on SST-2, the same as the run above.

In [ ]:
# NEW 12: one training run (build, train, score, save)
from peft import PeftModel
from transformers import PrinterCallback

def study_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "macro_f1": f1_score(labels, preds, average="macro")}

def build_model(s, num_labels, seed):
    """Frozen BERT + new classifier layer + LoRA as given in setting s."""
    set_seed(seed)
    # CLASSIFIER: BERT + a new linear layer on top ([CLS] -> BERT's pooler -> linear layer, 768 -> num_labels).
    # The linear layer starts random and is trained, BERT's own weights stay frozen.
    m = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)
    # >>> LoRA VARIATIONS go in here (r, modules, layers). Alpha keeps the ratio LORA_ALPHA / LORA_R of the config.
    lora = LoraConfig(task_type=TaskType.SEQ_CLS,      # SEQ_CLS keeps the new classifier layer trainable next to LoRA
                      r=s["r"], lora_alpha=s["r"] * LORA_ALPHA // LORA_R, lora_dropout=LORA_DROPOUT,
                      target_modules=s["modules"], layers_to_transform=s["layers"])
    return get_peft_model(m, lora)

def adapter_dir(dataset, setting, seed):
    return os.path.join(STUDY_DIR, "adapters", dataset, f"{setting.replace(' ', '_')}_seed{seed}")

def run_study(dataset, setting, seed):
    group, s = STUDY_SETTINGS[setting]
    d = STUDY_DATA[dataset]

    # 1. data, cut to the text length of this setting
    train_ds, dev_ds = tokenized(dataset, "train", s), tokenized(dataset, "dev", s)

    # 2. model: frozen BERT + classifier layer + LoRA of this setting
    model = build_model(s, d["num_labels"], seed)

    # 3. training settings of the trainable parts (LoRA A, B + classifier layer)
    total = STUDY_QUICK_STEPS if STUDY_QUICK else math.ceil(len(train_ds) / BATCH_SIZE) * STUDY_EPOCHS[dataset]
    curve = setting == "reference" and seed == STUDY_SEEDS[0]     # learning curve only for the reference
    eval_key = next(iter(eval_kw))                                  # same name trick as in the training setup above
    args = TrainingArguments(
        output_dir=os.path.join(STUDY_DIR, "tmp"),
        num_train_epochs=STUDY_EPOCHS[dataset],
        max_steps=STUDY_QUICK_STEPS if STUDY_QUICK else -1,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE * 2,
        learning_rate=s["lr"],                     # >>> learning rate of this setting
        weight_decay=WEIGHT_DECAY,
        warmup_steps=int(WARMUP_FRACTION * total),
        fp16=USE_FP16,
        logging_strategy="no",                    # no loss printouts, keeps the run output to one line per run
        disable_tqdm=True,
        dataloader_pin_memory=HAS_GPU,
        save_strategy="no",
        report_to="none",
        seed=seed,
        label_names=["labels"],
        remove_unused_columns=False,
        dataloader_num_workers=2,
        **({eval_key: "steps", "eval_steps": max(1, total // 5)} if curve else {eval_key: "no"}),
    )

    # 4. train
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                      data_collator=data_collator, compute_metrics=study_metrics)
    trainer.remove_callback(PrinterCallback)   # no training summary printout, one line per run is enough
    if HAS_GPU:
        torch.cuda.reset_peak_memory_stats()
    start = time.time()
    trainer.train()
    seconds = time.time() - start

    # 5. ACCURACY TEST: predicted class vs. true class on the dev texts
    out = trainer.predict(dev_ds, metric_key_prefix="eval")
    m = out.metrics
    pred_rows = [{"dataset": dataset, "setting": setting, "seed": seed, "text": i, "label": int(l), "pred": int(p)}
                 for i, (l, p) in enumerate(zip(out.label_ids, out.predictions.argmax(-1)))]

    # 6. save the adapter, return the scores
    trainer.model.save_pretrained(adapter_dir(dataset, setting, seed))
    curve_rows = [{"dataset": dataset, "step": h["step"], "accuracy": h["eval_accuracy"]} for h in trainer.state.log_history if "eval_accuracy" in h]
    result = {"dataset": dataset, "group": group, "setting": setting, "what_changed": what_changed(setting), "seed": seed,
              "accuracy": m["eval_accuracy"], "macro_f1": m["eval_macro_f1"], "seconds": seconds,
              "trainable": count_params(model)[1], "peak_gpu_gb": torch.cuda.max_memory_allocated() / 1e9 if HAS_GPU else 0.0}
    del trainer, model
    if HAS_GPU:
        torch.cuda.empty_cache()
    return result, curve_rows, pred_rows

# check: the reference must have as many trainable weights as the feasibility run above (296,450 on SST-2)
print("trainable weights (LoRA + classifier) on a 2-class dataset:")
for name in STUDY_SETTINGS:
    print(f"  {name:18} {count_params(build_model(STUDY_SETTINGS[name][1], 2, 0))[1]:>10,}")

## 13. NEW Baseline: BERT as it is

**Why:** to see how much LoRA adds at all. Frozen BERT, mean over the tokens of the last layer, a logistic regression on top (scikit-learn, no training steps). This is the grey line in the plots.

In [ ]:
# NEW 13: baseline, frozen BERT + logistic regression
from transformers import AutoModel
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DEVICE = "cuda" if HAS_GPU else "cpu"
frozen_bert = AutoModel.from_pretrained(MODEL_NAME, add_pooling_layer=False).to(DEVICE).eval()

@torch.no_grad()
def layer_vectors(model, texts):
    """One vector per text and layer (mean over the tokens), shape [13, texts, 768]. Layer 0 = embeddings."""
    tokenizer.truncation_side = "right"
    out, texts = [], list(texts)
    for i in range(0, len(texts), 64):
        b = tokenizer(texts[i:i + 64], truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(DEVICE)
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_FP16):
            hidden = model(**b, output_hidden_states=True).hidden_states
        mask = b["attention_mask"].unsqueeze(-1).float()
        out.append(torch.stack([(h.float() * mask).sum(1) / mask.sum(1) for h in hidden]).cpu())
    return torch.cat(out, 1).numpy()

baseline = {}
for name, d in STUDY_DATA.items():
    X, X_dev = layer_vectors(frozen_bert, d["train"].text)[-1], layer_vectors(frozen_bert, d["dev"].text)[-1]
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    best_c = GridSearchCV(clf, {"logisticregression__C": [0.001, 0.01, 0.1, 1]}, cv=3).fit(X[:5000], d["train"].label[:5000]).best_params_
    preds = clf.set_params(**best_c).fit(X, d["train"].label).predict(X_dev)
    baseline[name] = {"accuracy": accuracy_score(d["dev"].label, preds), "macro_f1": f1_score(d["dev"].label, preds, average="macro")}
baseline = pd.DataFrame(baseline).T
baseline

## 14. NEW Grid search: rank x number of LoRA layers

From `next_steps_studies.md`, Step 1. Rank and layers are varied **together**, learning rate is fixed.

| what | values |
|---|---|
| rank r | 1, 4, 8, 16 |
| LoRA in the last n blocks | 1, 2, 4, 8, 12 (12 = all) |
| learning rate | fixed, `GRID_LR` |
| seeds | 3 (`GRID_SEEDS`) |

20 combinations x 3 datasets x 3 seeds = 180 runs, roughly 4 to 6 hours on a T4. If the notebook stops, run the cell again, finished runs are skipped. Each run prints one line with what changed compared to the config.

In [ ]:
# NEW 14: grid search over rank x number of last layers (next_steps_studies.md, Step 1)
GRID_LR = 1e-4                                   # fixed in Step 1, see next_steps_studies.md
GRID_RANKS = [1, 4, 8, 16]
GRID_LAST_LAYERS = [1, 2, 4, 8, 12]              # LoRA in the last n of the 12 blocks, 12 = all
GRID_SEEDS = [0] if STUDY_QUICK else [0, 1, 2]   # 3 seeds, more = more time
GRID_DATASETS = STUDY_DATASETS

def last_layers(n):
    """Block numbers for the last n blocks. peft counts the blocks 0-11, so 'last 2' = [10, 11]."""
    return None if n == 12 else list(range(12 - n, 12))

for r in GRID_RANKS:
    for n in GRID_LAST_LAYERS:
        STUDY_SETTINGS[f"grid r{r} last{n}"] = ("grid", change(r=r, layers=last_layers(n), lr=GRID_LR))
grid_names = [n for n, (g, _) in STUDY_SETTINGS.items() if g == "grid"]

def run_many(names, datasets, seeds):
    """Run every setting x dataset x seed that is not done yet, so a stopped run can just be started again."""
    files = {k: os.path.join(STUDY_DIR, f"{k}.csv") for k in ("results", "curves", "predictions")}
    rows = {k: pd.read_csv(f).to_dict("records") if os.path.exists(f) else [] for k, f in files.items()}
    done = {(r["dataset"], r["setting"], r["seed"]) for r in rows["results"]}
    for dataset in datasets:
        for name in names:
            for seed in seeds:
                if (dataset, name, seed) in done:
                    continue
                result, curve_rows, pred_rows = run_study(dataset, name, seed)
                rows["results"].append(result)
                rows["curves"] += curve_rows
                rows["predictions"] += pred_rows
                for k, f in files.items():
                    pd.DataFrame(rows[k]).to_csv(f, index=False)
                print(f"{dataset:12} {name:20} seed {seed}  accuracy {result['accuracy']:.3f}  {result['seconds']:.0f}s   changed: {result['what_changed']}")
    return pd.DataFrame(rows["results"])

print(len(grid_names) * len(GRID_SEEDS) * len(GRID_DATASETS), "grid runs")
all_results = run_many(grid_names, GRID_DATASETS, GRID_SEEDS)

## 15. NEW Results and the default values to report

- **Heatmap per dataset**: mean dev accuracy over the seeds, rank against the number of LoRA blocks.
- **Best per dataset** and **best overall** (mean over the datasets), with the seed spread.
- **Cheapest within half a point** of the best overall: fewest trainable weights, often the better default.

`DEFAULTS` at the end is what goes into the layer study (`drafts/Draft_addLayerStudy.ipynb`).

In [ ]:
# NEW 15: results plots and tables
def save_plot(fig, name):
    plt.rcParams["svg.fonttype"] = "none"   # text stays editable in the SVG
    fig.tight_layout()
    for ext in ("png", "svg"):
        fig.savefig(os.path.join(STUDY_DIR, f"{name}.{ext}"), dpi=200, bbox_inches="tight")
    plt.show()

grid_table = all_results[all_results.setting.isin(grid_names)].copy()
grid_table["rank"] = grid_table.setting.map(lambda n: STUDY_SETTINGS[n][1]["r"])
grid_table["last_layers"] = grid_table.setting.map(lambda n: int(n.split("last")[1]))

# heatmap per dataset, mean over seeds
fig, axes = plt.subplots(1, len(GRID_DATASETS), figsize=(3.6 * len(GRID_DATASETS), 3), squeeze=False)
for ax, d in zip(axes[0], GRID_DATASETS):
    t = grid_table[grid_table.dataset == d].pivot_table(index="rank", columns="last_layers", values="accuracy")
    ax.imshow(t.values, cmap="viridis", aspect="auto")
    for i in range(t.shape[0]):
        for j in range(t.shape[1]):
            ax.text(j, i, f"{t.values[i, j]:.3f}", ha="center", va="center", fontsize=7, color="white")
    ax.set(title=f"{d} (baseline {baseline.accuracy[d]:.3f})", xticks=range(t.shape[1]), xticklabels=t.columns,
           yticks=range(t.shape[0]), yticklabels=t.index, xlabel="LoRA in the last n blocks", ylabel="rank r")
save_plot(fig, "grid_rank_x_layers")

# best per dataset, with the spread over seeds
per_dataset = (grid_table.groupby(["dataset", "rank", "last_layers"])
               .agg(accuracy=("accuracy", "mean"), seed_std=("accuracy", "std"), trainable=("trainable", "first")).reset_index())
best_per_dataset = per_dataset.loc[per_dataset.groupby("dataset").accuracy.idxmax()]
print("best per dataset:")
for _, r in best_per_dataset.iterrows():
    print(f"  {r.dataset:12} r={int(r['rank']):2}, last {int(r.last_layers):2} blocks  -> {r.accuracy:.3f} +- {r.seed_std:.3f}")

# best overall = mean over the datasets, and the cheapest combination close to it
overall = (grid_table.groupby(["rank", "last_layers"])
           .agg(accuracy=("accuracy", "mean"), trainable=("trainable", "mean")).reset_index()
           .sort_values("accuracy", ascending=False))
best = overall.iloc[0]
close = overall[overall.accuracy >= best.accuracy - 0.005].sort_values("trainable").iloc[0]
DEFAULTS = {"r": int(best["rank"]), "last_layers": int(best.last_layers), "lr": GRID_LR}
print(f"best overall: r={int(best['rank'])}, last {int(best.last_layers)} blocks, lr={GRID_LR}  -> mean dev accuracy {best.accuracy:.3f}")
print(f"cheapest within 0.5 points: r={int(close['rank'])}, last {int(close.last_layers)} blocks  -> {close.accuracy:.3f}, {int(close.trainable):,} trainable weights")
print("DEFAULTS for Step 2:", DEFAULTS, " (change by hand after the discussion)")
overall.head(8).round(4)

## 16. NEW Save everything for our discussion

`study/summary.md` has all tables, `study.zip` has everything: tables as CSV, plots as PNG and SVG, and the adapters of all runs.

In [ ]:
# NEW 16: save tables, summary and zip
!pip install -q tabulate
tables = {"data": data_table, "classes": class_table, "baseline": baseline.reset_index(names="dataset"),
          "grid_per_dataset": per_dataset, "grid_best_per_dataset": best_per_dataset, "grid_overall": overall}
summary = [f"# LoRA grid search\n\nreference: r={LORA_R}, alpha={LORA_ALPHA}, modules={LORA_TARGET_MODULES}, max length {MAX_LENGTH}, "
           f"grid lr {GRID_LR}, seeds {GRID_SEEDS}, quick run: {STUDY_QUICK}\n\nDEFAULTS for Step 2: {DEFAULTS}"]
for name, t in tables.items():
    t.to_csv(os.path.join(STUDY_DIR, f"{name}.csv"), index=False)
    summary.append(f"## {name}\n\n" + t.round(4).to_markdown(index=False))
summary.append("## Plots\n\n" + "\n".join(f"- {p}" for p in sorted(os.listdir(STUDY_DIR)) if p.endswith(".png")))
with open(os.path.join(STUDY_DIR, "summary.md"), "w") as f:
    f.write("\n\n".join(summary))
shutil.make_archive(STUDY_DIR, "zip", STUDY_DIR)
print("saved", STUDY_DIR + ".zip")